# Chapter 12b: Transformer Variants — Same Engine, Different Vehicles

The transformer architecture is like an engine.
You can put the same engine in a car, a boat, or a plane.
The engine (attention) is identical. What changes:

1. **Input**: how you convert raw data into vectors
2. **Masking**: what each token is allowed to see
3. **Training objective**: what you ask it to predict

That's it. Three knobs. Every transformer variant is just
a different combination of these three choices.

In [ ]:
import math
import random

def softmax(vals):
    mx = max(vals)
    exps = [math.exp(v - mx) for v in vals]
    s = sum(exps)
    return [e / s for e in exps]

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def mat_vec(M, v):
    return [sum(M[i][j] * v[j] for j in range(len(v))) for i in range(len(M))]

def self_attention(X, W_Q, W_K, W_V, mask=None):
    """Self-attention that works the same for text, images, audio, etc."""
    seq_len = len(X)
    d_k = len(W_Q[0])
    scale = math.sqrt(d_k)
    
    Qs = [mat_vec(W_Q, x) for x in X]
    Ks = [mat_vec(W_K, x) for x in X]
    Vs = [mat_vec(W_V, x) for x in X]
    
    outputs = []
    all_weights = []
    for i in range(seq_len):
        scores = [dot(Qs[i], Ks[j]) / scale for j in range(seq_len)]
        
        if mask is not None:
            for j in range(seq_len):
                if mask[i][j] == 0:
                    scores[j] = -1e9
        
        weights = softmax(scores)
        all_weights.append(weights)
        
        out = [0.0] * d_k
        for j in range(seq_len):
            for d in range(d_k):
                out[d] += weights[j] * Vs[j][d]
        outputs.append(out)
    
    return outputs, all_weights

print("self_attention() defined.")
print("The SAME function runs for text, images, audio, protein...")
print("Only the input vectors and mask change.")

---
## Knob 1: Input — How Raw Data Becomes Vectors

Attention only knows vectors. Every domain must convert its raw data
into a **sequence of vectors** before attention can process it.

```
Text:    "the cat sat"    -> tokenize -> embed -> [vec, vec, vec]
Image:   224x224 pixels   -> split into 16x16 patches -> project -> [vec, vec, ...]
Audio:   3 seconds of wav -> spectrogram -> frames -> project -> [vec, vec, ...]
Protein: MKFLV...         -> each amino acid -> embed -> [vec, vec, ...]
```

Once you have vectors, attention doesn't know or care what they came from.

In [ ]:
random.seed(42)

d_model = 4

print("How different domains create input vectors:")
print("=" * 60)

# --- Text (GPT / BERT) ---
print("\n1. TEXT (GPT, BERT, Claude)")
print("   Raw: 'the cat sat'")
print("   Step 1: Tokenize -> ['the', 'cat', 'sat']")
print("   Step 2: Embedding lookup (learned table)")

text_vocab = {
    "the": [0.1, 0.3, 0.8, 0.2],
    "cat": [0.8, 0.1, 0.3, 0.9],
    "sat": [0.7, 0.2, 0.5, 0.8],
}
text_tokens = ["the", "cat", "sat"]
text_vectors = [text_vocab[t] for t in text_tokens]

for tok, vec in zip(text_tokens, text_vectors):
    print(f'   "{tok}" -> {vec}')
print(f"   Result: {len(text_vectors)} vectors, each {d_model}-dim")

# --- Image (ViT) ---
print("\n2. IMAGE (Vision Transformer / ViT)")
print("   Raw: 224x224 pixel image")
print("   Step 1: Split into 16x16 patches")

img_size = 224
patch_size = 16
n_patches = (img_size // patch_size) ** 2

print(f"   224 / 16 = {img_size // patch_size} patches per side")
print(f"   {img_size // patch_size} x {img_size // patch_size} = {n_patches} patches total")
print("   Step 2: Flatten each patch (16x16x3 = 768 pixels)")
print(f"   Step 3: Linear projection (768 -> {d_model})")

image_vectors = [[random.gauss(0, 0.3) for _ in range(d_model)] for _ in range(4)]
patch_labels = ["top-left", "top-right", "bot-left", "bot-right"]
for label, vec in zip(patch_labels, image_vectors):
    print(f"   patch({label}) -> [{', '.join(f'{v:.2f}' for v in vec)}]")
print(f"   Result: {n_patches} vectors (but showing 4), each {d_model}-dim")

# --- Audio (Whisper) ---
print("\n3. AUDIO (Whisper)")
print("   Raw: 3 seconds of speech at 16kHz = 48,000 samples")
print("   Step 1: Convert to mel spectrogram (80 freq bins x 150 frames)")
print("   Step 2: Two 1D convolutions -> downsample to ~75 frames")
print(f"   Step 3: Linear projection -> {d_model}-dim vectors")

audio_vectors = [[random.gauss(0, 0.3) for _ in range(d_model)] for _ in range(4)]
for i, vec in enumerate(audio_vectors):
    ms = i * 40
    print(f"   frame({ms}ms) -> [{', '.join(f'{v:.2f}' for v in vec)}]")
print(f"   Result: ~75 vectors, each {d_model}-dim")

# --- Protein (AlphaFold / ESM) ---
print("\n4. PROTEIN (AlphaFold, ESM)")
print("   Raw: amino acid sequence 'MKFLVTL...'")
print("   Step 1: Each amino acid -> embedding (20 possible + specials)")

amino_acids = ["M", "K", "F", "L"]
protein_vectors = [[random.gauss(0, 0.3) for _ in range(d_model)] for _ in range(4)]
for aa, vec in zip(amino_acids, protein_vectors):
    print(f"   {aa} -> [{', '.join(f'{v:.2f}' for v in vec)}]")
print(f"   Result: 1 vector per amino acid, each {d_model}-dim")

In [ ]:
# The key point: once you have vectors, attention is IDENTICAL

random.seed(42)
W_Q = [[random.gauss(0, 0.3) for _ in range(3)] for _ in range(d_model)]
W_K = [[random.gauss(0, 0.3) for _ in range(3)] for _ in range(d_model)]
W_V = [[random.gauss(0, 0.3) for _ in range(3)] for _ in range(d_model)]

print("Same attention function, different inputs:")
print("=" * 55)

domains = [
    ("Text (GPT)", text_vectors, ["the", "cat", "sat"]),
    ("Image (ViT)", image_vectors, ["patch1", "patch2", "patch3", "patch4"]),
    ("Audio (Whisper)", audio_vectors, ["0ms", "40ms", "80ms", "120ms"]),
    ("Protein (ESM)", protein_vectors, ["M", "K", "F", "L"]),
]

for name, vectors, labels in domains:
    outputs, weights = self_attention(vectors, W_Q, W_K, W_V)
    print(f"\n  {name}:")
    print(f"    Input: {len(vectors)} vectors of dim {len(vectors[0])}")
    print(f"    Attention weights (who attends to whom):")
    
    header = "          "
    for l in labels:
        header += f" {l:>6}"
    print(f"    {header}")
    
    for i, l in enumerate(labels):
        row = f"    {l:>10}"
        for j in range(len(labels)):
            pct = int(weights[i][j] * 100)
            row += f" {pct:>5}%"
        print(row)

print("\n  The attention function has NO idea what the vectors represent.")
print("  Text, pixels, sound, amino acids — all just vectors to it.")

---
## Knob 2: Masking — What Each Token Can See

The mask determines which positions each token is allowed to attend to.
This single choice creates fundamentally different model behaviors.

```
No mask (Encoder / Bidirectional):   sees EVERYTHING
Causal mask (Decoder):               sees only PAST
Cross-attention mask (Enc-Dec):      decoder sees encoder output
```

In [ ]:
tokens = ["the", "cat", "sat", "down"]
n = len(tokens)

print("Three masking strategies:")
print("=" * 60)

# --- Encoder (no mask) ---
print("\n1. ENCODER (BERT, ViT) — no mask, sees everything")
print("   Used for: understanding, classification, search")
print()

encoder_mask = [[1] * n for _ in range(n)]

header = "          "
for t in tokens:
    header += f" {t:>5}"
print(f"   {header}")
for i, t in enumerate(tokens):
    row = f"   {t:>10}"
    for j in range(n):
        row += "    ok" if encoder_mask[i][j] else "     X"
    print(row)
print("   Every token sees every other token (bidirectional).")

# --- Decoder (causal mask) ---
print("\n2. DECODER (GPT, Claude) — causal mask, sees only past")
print("   Used for: text generation")
print()

decoder_mask = [[1 if j <= i else 0 for j in range(n)] for i in range(n)]

print(f"   {header}")
for i, t in enumerate(tokens):
    row = f"   {t:>10}"
    for j in range(n):
        row += "    ok" if decoder_mask[i][j] else "  MASK"
    print(row)
print("   Each token only sees itself and previous tokens.")

# --- Encoder-Decoder ---
print("\n3. ENCODER-DECODER (T5, original Transformer)")
print("   Used for: translation, summarization")
print()
print("   Encoder side: no mask (sees full input)")
print("   Decoder side: causal mask (generates left to right)")
print("   Cross-attention: decoder queries attend to all encoder outputs")
print()
print('   Example: translate "the cat sat" -> "le chat assis"')
print()

enc_tokens = ["the", "cat", "sat"]
dec_tokens = ["le", "chat", "assis"]

print("   Cross-attention mask (decoder -> encoder):")
cross_header = "          "
for t in enc_tokens:
    cross_header += f" {t:>5}"
print(f"   {cross_header}  (encoder)")
for t in dec_tokens:
    row = f"   {t:>10}"
    for _ in enc_tokens:
        row += "    ok"
    print(row)
print("   (decoder)")
print("   Each decoder token can see ALL encoder tokens.")

In [ ]:
# Run attention with different masks on the SAME data

X = text_vectors  # reuse ["the", "cat", "sat"]
n = len(X)

print("Same input, different masks -> different attention patterns:")
print("=" * 55)

# Encoder mask (see everything)
enc_mask = [[1] * n for _ in range(n)]
_, enc_weights = self_attention(X, W_Q, W_K, W_V, mask=enc_mask)

# Decoder mask (causal)
dec_mask = [[1 if j <= i else 0 for j in range(n)] for i in range(n)]
_, dec_weights = self_attention(X, W_Q, W_K, W_V, mask=dec_mask)

labels = ["the", "cat", "sat"]

for name, weights in [("Encoder (no mask)", enc_weights), ("Decoder (causal)", dec_weights)]:
    print(f"\n  {name}:")
    header = "          "
    for l in labels:
        header += f" {l:>6}"
    print(f"    {header}")
    for i, l in enumerate(labels):
        row = f"    {l:>10}"
        for j in range(n):
            pct = int(weights[i][j] * 100)
            row += f" {pct:>5}%"
        print(row)

print("\n  Notice: 'the' in the encoder attends to all 3 tokens.")
print("  But 'the' in the decoder attends only to itself (100%).")
print("  The mask changes the behavior without changing the algorithm.")

---
## Knob 3: Training Objective — What to Predict

The training objective determines what the model **learns** from data.
Same architecture, different objective -> completely different model.

| Model | Objective | What it learns |
|-------|-----------|----------------|
| GPT   | Predict next token | Text generation |
| BERT  | Predict masked token | Text understanding |
| ViT   | Predict image class | Image classification |
| CLIP  | Match image to text | Cross-modal search |
| Whisper | Predict text from audio | Speech recognition |
| MAE   | Reconstruct masked patches | Image understanding |

In [ ]:
print("Training objectives compared:")
print("=" * 65)

print()
print('1. GPT (next token prediction):')
print('   Input:  "The cat sat"')
print('   Target: "cat sat down"  (shifted by 1)')
print('   Loss: how well did it predict each next token?')
print()
print('   "The"     -> predict "cat"      (learn: animals follow "the")')
print('   "The cat" -> predict "sat"      (learn: verbs follow nouns)')
print()

print('2. BERT (masked token prediction):')
print('   Input:  "The [MASK] sat down"')
print('   Target: "cat" (the masked word)')
print('   Loss: how well did it predict the hidden word?')
print()
print('   It can see BOTH sides: "The __ sat down"')
print('   "sat" gives a clue (something that sits)')
print('   "down" gives a clue (something that sits down)')
print('   Bidirectional context -> better understanding')
print()

print('3. ViT (image classification):')
print('   Input:  [patch1, patch2, ..., patch196] + [CLS] token')
print('   Target: "cat" (image label)')
print('   Loss: did [CLS] token predict the right class?')
print()
print('   The [CLS] token attends to all patches.')
print('   It learns to summarize the whole image.')
print()

print('4. CLIP (image-text matching):')
print('   Input:  image patches + text tokens (separate encoders)')
print('   Target: which image goes with which text?')
print('   Loss: cosine similarity between matched pairs')
print()
print('   Photo of a cat  <-> "a photo of a cat"     (match)')
print('   Photo of a cat  <-> "a red sports car"     (no match)')

In [ ]:
# Simulate GPT vs BERT on the same sentence

sentence = ["The", "cat", "sat", "down"]

print("GPT vs BERT: same sentence, different training")
print("=" * 55)

print("\nGPT (decoder, causal mask, predict NEXT):")
print()
for i in range(len(sentence) - 1):
    context = sentence[:i+1]
    target = sentence[i+1]
    visible = " ".join(context)
    hidden = " ".join(["___"] * (len(sentence) - i - 1))
    print(f"  sees: [{visible}] {hidden}")
    print(f"  predicts: '{target}'")
    print()

print("BERT (encoder, no mask, predict MASKED):")
print()
for i in range(len(sentence)):
    masked = list(sentence)
    target = masked[i]
    masked[i] = "[MASK]"
    print(f"  sees: [{' '.join(masked)}]")
    print(f"  predicts: '{target}' (the masked word)")
    print()

print("GPT:  can only use LEFT context (past). Good for generation.")
print("BERT: uses BOTH sides (past + future). Good for understanding.")
print("      But BERT can't generate text — it would need to see")
print("      the future, which doesn't exist yet during generation.")

---
## Vision Transformer (ViT) — Images as Token Sequences

The key insight: an image is just a grid of patches.
Treat each patch like a "word" and run standard attention.

```
Image (224 x 224):
  +----+----+----+
  | p1 | p2 | p3 |     Each patch = 16x16 pixels
  +----+----+----+     = 16 * 16 * 3 (RGB) = 768 numbers
  | p4 | p5 | p6 |     Linear projection: 768 -> d_model
  +----+----+----+
  | p7 | p8 | p9 |     9 patches = 9 "tokens"
  +----+----+----+     (real ViT: 196 patches)

  Then: self_attention([p1, p2, ..., p9])  <- same function!
```

In [ ]:
# Simulate Vision Transformer on a tiny 6x6 "image"

random.seed(7)

# 6x6 image, 2x2 patches -> 9 patches
img_size = 6
patch_size = 2
patches_per_side = img_size // patch_size
n_patches = patches_per_side ** 2

# Simulate a simple image: bright top-left (sky), dark bottom (ground)
image = []
for r in range(img_size):
    row = []
    for c in range(img_size):
        if r < 3:
            row.append(0.9 + random.gauss(0, 0.05))  # bright (sky)
        else:
            row.append(0.2 + random.gauss(0, 0.05))  # dark (ground)
    image.append(row)

print("Simulated 6x6 image (bright=sky, dark=ground):")
print()
for r in range(img_size):
    row_str = "  "
    for c in range(img_size):
        val = image[r][c]
        if val > 0.7:
            row_str += " ##"
        elif val > 0.4:
            row_str += " .."
        else:
            row_str += " __"
    label = "  (sky)" if r < 3 else "  (ground)"
    print(row_str + label)

# Extract patches
print(f"\nSplit into {patches_per_side}x{patches_per_side} = {n_patches} patches of {patch_size}x{patch_size}:")
print()

patches = []
patch_labels = []
for pr in range(patches_per_side):
    for pc in range(patches_per_side):
        patch_pixels = []
        for r in range(patch_size):
            for c in range(patch_size):
                patch_pixels.append(image[pr*patch_size+r][pc*patch_size+c])
        patches.append(patch_pixels)
        avg = sum(patch_pixels) / len(patch_pixels)
        label = f"p{pr*patches_per_side+pc+1}"
        region = "sky" if pr < patches_per_side // 2 + 1 and avg > 0.5 else "ground"
        patch_labels.append(f"{label}({region})")
        print(f"  {label}: pixels={[f'{p:.2f}' for p in patch_pixels]}  avg={avg:.2f}  ({region})")

print(f"\nEach patch has {patch_size*patch_size} pixel values.")
print(f"In real ViT: each patch has 16*16*3 = 768 values,")
print(f"projected to d_model (e.g. 768) via a linear layer.")

In [ ]:
# Run attention on image patches

random.seed(42)
d_patch = len(patches[0])  # 4 (2x2 pixels)
d_k = 3

W_Q_img = [[random.gauss(0, 0.4) for _ in range(d_k)] for _ in range(d_patch)]
W_K_img = [[random.gauss(0, 0.4) for _ in range(d_k)] for _ in range(d_patch)]
W_V_img = [[random.gauss(0, 0.4) for _ in range(d_k)] for _ in range(d_patch)]

# No mask — encoder style (ViT sees all patches)
_, img_weights = self_attention(patches, W_Q_img, W_K_img, W_V_img)

print("ViT attention: which patches attend to which?")
print("(encoder = no mask, every patch sees every other patch)")
print()

short_labels = [f"p{i+1}" for i in range(n_patches)]
header = "        "
for l in short_labels:
    header += f" {l:>4}"
print(header)

for i in range(n_patches):
    row = f"  {short_labels[i]:>6}"
    for j in range(n_patches):
        pct = int(img_weights[i][j] * 100)
        row += f" {pct:>3}%"
    # Find which patch this one attends to most
    max_j = img_weights[i].index(max(img_weights[i]))
    print(f"{row}  -> most: {short_labels[max_j]}")

print()
print("  Sky patches (p1-p5) tend to attend to other sky patches.")
print("  Ground patches (p6-p9) tend to attend to ground patches.")
print("  Similar regions find each other through attention!")
print()
print("  In a trained ViT, this is how the model learns:")
print("    'these patches belong to the same object (a cat)'")
print("    'these patches are background'")

---
## CLIP — Connecting Images and Text

CLIP uses **two** transformers: one for images, one for text.
It trains them so that matching image-text pairs have similar vectors.

```
Image of a cat  --->  Image Transformer  --->  vector A
"a photo of cat" --->  Text Transformer  --->  vector B

Training: make cosine_similarity(A, B) high for matching pairs
          make cosine_similarity(A, B) low for non-matching pairs
```

After training, you can:
- Search images with text ("find photos of cats")
- Classify images with any text label (zero-shot classification)

In [ ]:
# Simulate CLIP: image and text encoders produce comparable vectors

def cosine_sim(a, b):
    dot_val = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot_val / (mag_a * mag_b + 1e-8)

# Pretend these are the outputs of trained encoders
# (in reality: 512 or 768 dimensions)
image_embeddings = {
    "photo of cat":   [0.9, 0.1, 0.0, 0.2],
    "photo of dog":   [0.1, 0.9, 0.0, 0.3],
    "photo of car":   [0.0, 0.1, 0.9, 0.1],
    "photo of sunset": [0.2, 0.0, 0.1, 0.9],
}

text_embeddings = {
    "a cute kitten":     [0.85, 0.15, 0.05, 0.1],
    "a golden retriever": [0.15, 0.85, 0.05, 0.2],
    "a red sports car":  [0.05, 0.1, 0.88, 0.05],
    "a beautiful sunset": [0.15, 0.05, 0.1, 0.88],
}

print("CLIP: match images to text via cosine similarity")
print("=" * 60)
print()

print(f"  {'':>20}", end="")
for text in text_embeddings:
    short = text[:12]
    print(f" {short:>13}", end="")
print()
print(f"  {'':>20}", end="")
print(f" {'-'*13}" * len(text_embeddings))

for img_name, img_vec in image_embeddings.items():
    print(f"  {img_name:>20}", end="")
    best_score = -1
    best_text = ""
    for text_name, text_vec in text_embeddings.items():
        sim = cosine_sim(img_vec, text_vec)
        if sim > best_score:
            best_score = sim
            best_text = text_name
        marker = " ***" if sim > 0.9 else "    "
        print(f" {sim:>9.3f}{marker}", end="")
    print()

print()
print("  *** = best match")
print("  The diagonal is strong — matching pairs have high similarity.")
print()
print("  This is how image search works:")
print('  Query: "a cute kitten" -> compute text vector -> find nearest image')

---
## Whisper — Audio as Token Sequences

Whisper uses an **encoder-decoder** transformer:
- Encoder: processes audio (mel spectrogram frames)
- Decoder: generates text tokens (autoregressive)

```
Audio waveform
  -> mel spectrogram (80 frequency bins x N time frames)
  -> 2 conv layers (downsample)
  -> encoder (bidirectional attention on audio frames)
  -> decoder (causal attention + cross-attention to encoder)
  -> text tokens
```

In [ ]:
print("Whisper: Audio -> Text using Encoder-Decoder Transformer")
print("=" * 60)
print()
print("Input: 3 seconds of someone saying 'hello world'")
print()

# Simulate mel spectrogram (simplified)
print("Step 1: Convert waveform to mel spectrogram")
print("  80 frequency bins x 150 time frames")
print()
print("  Freq ^")
print("  high |          ...        ......")
print("       |      .........   ..........")
print("       |   ............  ...........")
print("       | ...............  ..........")
print("  low  |...............................")
print("       +------------------------------> Time")
print("        'hhhh' 'eee'  'lll' 'ooo'  'www' 'rrr' 'lll' 'ddd'")
print()

print("Step 2: Convolutions downsample to ~75 frames")
print("  Each frame becomes a vector (like a token)")
print()
print("Step 3: Encoder processes all frames (bidirectional)")
print("  Frame 1 can attend to frame 75 (no causal mask)")
print("  The encoder understands the full audio context")
print()
print("Step 4: Decoder generates text autoregressively")
print('  [START] -> "hello"')
print('  [START] "hello" -> "world"')
print('  [START] "hello" "world" -> [END]')
print()
print("  The decoder uses cross-attention to look at encoder output.")
print("  When generating 'world', it attends to the audio frames")
print("  that correspond to the 'w-r-l-d' sound.")

---
## The Complete Comparison

Every transformer variant is just three choices:

In [ ]:
print("Transformer Variants: Three Knobs")
print("=" * 70)
print()

variants = [
    ("GPT / Claude",  "text tokens",     "causal (decoder)",    "next token"),
    ("BERT",          "text tokens",     "none (encoder)",      "masked token"),
    ("ViT",           "image patches",   "none (encoder)",      "image class"),
    ("CLIP",          "patches + tokens","none (two encoders)", "image-text match"),
    ("Whisper",       "audio frames",    "enc-dec",             "audio -> text"),
    ("T5 / BART",     "text tokens",     "enc-dec",             "text -> text"),
    ("AlphaFold",     "amino acids",     "none + pair repr.",   "3D structure"),
    ("Stable Diff.",  "noise + text",    "cross-attn",          "denoise image"),
    ("Music Transf.", "audio tokens",    "causal (decoder)",    "next audio token"),
    ("CodeBERT",      "code tokens",     "none (encoder)",      "masked token"),
]

print(f"  {'Model':<16} {'Input':<18} {'Masking':<20} {'Objective'}")
print(f"  {'-'*16} {'-'*18} {'-'*20} {'-'*20}")
for name, inp, mask, obj in variants:
    print(f"  {name:<16} {inp:<18} {mask:<20} {obj}")

print()
print("  The attention mechanism is IDENTICAL in every row.")
print("  Q, K, V, softmax, weighted sum — same math everywhere.")

---
## Building a Domain-Specific Transformer

If you want to build a transformer for YOUR domain,
you make the same three decisions:

In [ ]:
print("Design Your Own Transformer: Decision Checklist")
print("=" * 65)
print()

examples = [
    {
        "domain": "Medical Report Generator",
        "input": "X-ray image patches + patient history tokens",
        "mask": "Encoder-decoder (encode image, generate report)",
        "objective": "Generate diagnostic text from image",
        "data": "Radiology images paired with reports",
    },
    {
        "domain": "Stock Price Predictor",
        "input": "Daily price/volume as vectors (each day = 1 token)",
        "mask": "Causal (can't see future prices)",
        "objective": "Predict next day's direction",
        "data": "Historical price sequences",
    },
    {
        "domain": "DNA Sequence Analyzer",
        "input": "Nucleotides A/T/G/C as tokens",
        "mask": "None (encoder, see full sequence)",
        "objective": "Predict function of gene region",
        "data": "Annotated genome sequences",
    },
    {
        "domain": "Legal Contract Reviewer",
        "input": "Legal text tokens (domain-specific tokenizer)",
        "mask": "None (encoder, see full document)",
        "objective": "Classify clause type / flag risks",
        "data": "Labeled legal documents",
    },
]

for ex in examples:
    print(f"  --- {ex['domain']} ---")
    print(f"  1. INPUT:     {ex['input']}")
    print(f"  2. MASKING:   {ex['mask']}")
    print(f"  3. OBJECTIVE: {ex['objective']}")
    print(f"  4. DATA:      {ex['data']}")
    print()

In [ ]:
# Decision flowchart

print("Decision Flowchart: What Kind of Transformer Do You Need?")
print("=" * 60)
print()
print("  Q1: Do you need to GENERATE output sequence?")
print("  |")
print("  +-- YES: Do you have a separate input to condition on?")
print("  |   |")
print("  |   +-- YES -> ENCODER-DECODER")
print("  |   |         (translation, image captioning, speech-to-text)")
print("  |   |")
print("  |   +-- NO  -> DECODER ONLY")
print("  |             (text generation, code completion, chatbot)")
print("  |")
print("  +-- NO: You need to UNDERSTAND / CLASSIFY input")
print("      |")
print("      +-> ENCODER ONLY")
print("          (classification, search, similarity, NER)")
print()
print("  Q2: Is your input NOT text?")
print("  |")
print("  +-- Image  -> split into patches, linear projection")
print("  +-- Audio  -> mel spectrogram, conv layers")
print("  +-- Tabular -> each row/feature as a token")
print("  +-- Graph  -> each node as a token + edge info")
print("  +-- Custom -> define your own tokenization")
print()
print("  After these choices, the attention mechanism is standard.")

In [ ]:
# The alternative: fine-tune instead of building from scratch

print("Build vs Fine-tune: What's Practical?")
print("=" * 60)
print()
print("  Option A: Train from scratch")
print("    - You define everything (tokenizer, architecture, data)")
print("    - Needs massive data (millions+ examples)")
print("    - Needs massive compute (days/weeks on many GPUs)")
print("    - When: novel domain with no existing model")
print("            (protein folding, chip design, weather)")
print()
print("  Option B: Fine-tune an existing model")
print("    - Start with pre-trained weights (GPT, BERT, ViT, etc.)")
print("    - Add domain data on top (thousands of examples)")
print("    - Needs modest compute (hours on 1-4 GPUs)")
print("    - When: your domain uses text, images, or audio")
print("            (legal, medical, finance, customer support)")
print()
print("  Option C: Prompt engineering / RAG")
print("    - Use existing LLM as-is, provide context in prompt")
print("    - No training at all")
print("    - When: you need good-enough results quickly")
print()
print("  Most practical for most people: B or C.")
print("  Training from scratch is for companies with")
print("  novel data types and large compute budgets.")

---
## Summary

| Knob | Choice | Effect |
|------|--------|--------|
| **Input** | How to vectorize raw data | Determines what domain the model works on |
| **Masking** | What each position can see | Determines encoder vs decoder vs both |
| **Objective** | What to predict during training | Determines what the model learns to do |

The attention mechanism itself — `softmax(QK^T / sqrt(d_k)) V` — is **identical**
across GPT, BERT, ViT, Whisper, AlphaFold, CLIP, and every other transformer.

```
GPT    = text tokens   + causal mask  + next token prediction
BERT   = text tokens   + no mask      + masked token prediction
ViT    = image patches + no mask      + image classification
CLIP   = patches+text  + no mask      + image-text matching
Whisper = audio frames + enc-dec mask + speech-to-text
```

To build your own: decide input format, masking strategy, and training objective.
The rest is standard transformer code.